# Loan Terms Assistant — QLoRA Fine-Tuning (Colab, free T4)

This notebook fine-tunes a small open-source model (**Qwen2.5-1.5B-Instruct**) with **QLoRA** so it behaves like a
loan/credit Terms & Conditions assistant: it answers only from the 5 provided bank documents, **cites** the
document + page, **refuses** off-topic questions, and says **"Not stated in the terms."** when the docs don't answer.

**How to use this notebook:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Run the cells top to bottom.
3. When asked, upload `train.jsonl` and `eval.jsonl` (from your project files).
4. Section 3 shows the **BEFORE** answers (base model). Section 6 shows the **AFTER** answers (fine-tuned).
   Screenshot both for at least: 1 answerable question, 1 off-topic question, 1 "not stated" question.
5. Section 7 launches a public Gradio demo link — that's your submission link.

If Colab crashes: read only the **first** red line. `out of memory` → lower batch size / max_seq_length below.
`unexpected keyword` / `cannot import` → restart runtime (Runtime → Restart session) and re-run the install cell.


## 1. Install libraries

In [ ]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.2 MB/s eta 0:00:00


In [ ]:
import torch, json
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - go to Runtime > Change runtime type > T4 GPU")


GPU available: True
GPU name: Tesla T4


## 2. Upload your data

Upload `train.jsonl` (the dataset you were given / built) and `eval.jsonl` (held-out questions).


In [ ]:
from google.colab import files
print("Upload train.jsonl and eval.jsonl now:")
uploaded = files.upload()


Upload train.jsonl and eval.jsonl now:


Saving eval.jsonl to eval.jsonl


In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

train_data = load_jsonl("train.jsonl")
eval_data = load_jsonl("eval.jsonl")
print("train examples:", len(train_data))
print("eval examples:", len(eval_data))
print()
print("Example:")
print(json.dumps(train_data[0], indent=2, ensure_ascii=False))


train examples: 45
eval examples: 8

Example:
{
  "messages": [
    {
      "role": "system",
      "content": "You are a loan/credit terms assistant. Answer based ONLY on the provided documents. Cite the source PDF. If off-topic, refuse. If not in docs, say 'Not stated in the terms.'"
    },
    {
      "role": "user",
      "content": "What happens if I miss an EMI payment?"
    },
    {
      "role": "assistant",
      "content": "Missing an EMI payment constitutes an Event of Default. The Bank may recall the whole advance and enforce its security upon the happening of such event. (south_indian_bank_loan.pdf)"
    }
  ]
}


## 3. Load the base model in 4-bit (QLoRA) + BEFORE answers

We load Qwen2.5-1.5B-Instruct with its weights squeezed to 4-bit (this is the "Q" in QLoRA — see HOW_IT_WORKS.html).
Then, **before any training**, we ask it the eval questions to record the BEFORE behaviour.


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
print("Base model loaded in 4-bit.")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base model loaded in 4-bit.


In [ ]:
SYSTEM_PROMPT = ("You are a loan/credit Terms & Conditions assistant. You answer ONLY questions about the loan "
                  "documents you were trained on. You cite the document and page. If something is not in the terms "
                  "you say 'Not stated in the terms.' You refuse anything off-topic or advice.")

def ask(model, tokenizer, question, max_new_tokens=150):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip()


In [ ]:
# The 8 held-out questions from eval.jsonl, PLUS 2 extra probes for the two behaviours
# eval.jsonl doesn't cover (off-topic refusal, and "not stated in the terms").
# These two are NOT in train.jsonl, so they're a genuine held-out check of learned behaviour.
demo_questions = [q["messages"][1]["content"] for q in eval_data]
demo_questions.append("Can you recommend a good movie to watch tonight?")          # off-topic -> should refuse
demo_questions.append("Does the bank offer any cashback rewards for early loan repayment?")  # not-in-docs -> "Not stated in the terms."

before_answers = {}
for q in demo_questions:
    before_answers[q] = ask(model, tokenizer, q)

for q, a in before_answers.items():
    print("Q:", q)
    print("BEFORE:", a)
    print("-" * 80)


Q: Can the bank demand payment from me as a guarantor before taking legal action against the main borrower?
BEFORE: The terms of your loan agreement should clearly state whether the bank can demand payment from you as a guarantor before taking legal action against the main borrower. The exact language will depend on the specific terms of your contract, but it typically involves provisions that allow the lender to seek repayment from the guarantor if the main borrower fails to meet their obligations.

If there is no such provision in your contract, then the bank would need to take legal action against both parties (the guarantor and the main borrower) separately before proceeding with any demands for repayment. This means they could only demand payment from you after the main borrower has been legally pursued by them.
--------------------------------------------------------------------------------
Q: Who is responsible for paying the various costs, taxes, and legal expenses associated w

## 4. Prepare QLoRA (LoRA adapters on the 4-bit base)

We freeze the 4-bit base and add small trainable adapters (see HOW_IT_WORKS.html — the A×B matrices).


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # should be roughly ~1% of total params


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 5. Fine-tune with QLoRA

In [ ]:
from datasets import Dataset

# trl >= 1.0 "messages" columnni o'zi tushunadi, chat template'ni o'zi qo'llaydi
train_dataset = Dataset.from_list(train_data)
print(train_dataset[0])

{'messages': [{'role': 'system', 'content': "You are a loan/credit terms assistant. Answer based ONLY on the provided documents. Cite the source PDF. If off-topic, refuse. If not in docs, say 'Not stated in the terms.'"}, {'role': 'user', 'content': 'What happens if I miss an EMI payment?'}, {'role': 'assistant', 'content': 'Missing an EMI payment constitutes an Event of Default. The Bank may recall the whole advance and enforce its security upon the happening of such event. (south_indian_bank_loan.pdf)'}]}


In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./qlora-loan-assistant",
    num_train_epochs=4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    max_length=768,
    assistant_only_loss=True,
    report_to="none",
      bf16=True,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

trainer.train()

Tokenizing train dataset:   0%|          | 0/45 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/45 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/45 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/45 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,2.280215
10,1.717348
15,1.357877
20,1.030559


TrainOutput(global_step=24, training_loss=1.5020925601323445, metrics={'train_runtime': 140.4395, 'train_samples_per_second': 1.282, 'train_steps_per_second': 0.171, 'total_flos': 177363626366976.0, 'train_loss': 1.5020925601323445, 'entropy': 1.106284495194753, 'num_tokens': 20512.0, 'mean_token_accuracy': 0.7419822454452515, 'epoch': 4.0})

In [ ]:
ADAPTER_DIR = "./qlora-loan-assistant-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Adapter saved to", ADAPTER_DIR)


Adapter saved to ./qlora-loan-assistant-adapter


## 6. AFTER — same questions, fine-tuned model

`model` already has the trained adapter attached (we never removed it), so we can reuse `ask()` directly.


In [ ]:
model.eval()
after_answers = {}
for q in demo_questions:
    after_answers[q] = ask(model, tokenizer, q)

for q in demo_questions:
    print("Q:", q)
    print("BEFORE:", before_answers[q])
    print("AFTER :", after_answers[q])
    print("=" * 100)


Q: Can the bank demand payment from me as a guarantor before taking legal action against the main borrower?
BEFORE: The terms of your loan agreement should clearly state whether the bank can demand payment from you as a guarantor before taking legal action against the main borrower. The exact language will depend on the specific terms of your contract, but it typically involves provisions that allow the lender to seek repayment from the guarantor if the main borrower fails to meet their obligations.

If there is no such provision in your contract, then the bank would need to take legal action against both parties (the guarantor and the main borrower) separately before proceeding with any demands for repayment. This means they could only demand payment from you after the main borrower has been legally pursued by them.
AFTER : Yes. The Bank may at any time, without prejudice to other rights available to it under these conditions, require the Guarantor to pay immediately all sums owing by

**Screenshot at least 3 BEFORE/AFTER pairs from the cell above** — one answerable question, the off-topic one
("recommend a movie"), and the not-in-docs one ("cashback rewards") — for your submission.


## 7. Deploy a live demo with Gradio

In [ ]:
import gradio as gr

def chat_fn(message, history):
    return ask(model, tokenizer, message)

demo = gr.ChatInterface(
    fn=chat_fn,
    title="Loan Terms Assistant (QLoRA fine-tuned Qwen2.5-1.5B)",
    description="Ask about CIBC, CIMB, NBU, South Indian Bank, or Standard Chartered personal loan terms.",
)

demo.launch(share=True)  # gives you a public gradio.live link - this is your demo link to submit


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0df24e501eacc6f412.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 8. Fine-tuning vs RAG — draft answer (put this in your own words for submission)

Fine-tuning changes the model's **weights** (here, a small set of LoRA adapter weights) so the *behaviour* —
tone, format, when to refuse, when to say "Not stated in the terms" — becomes baked into the model itself. It's
slow and needs a training run, but at inference time it's just a normal model call: no extra retrieval step,
and the behaviour is consistent even for questions phrased very differently from the training examples.

RAG (Retrieval-Augmented Generation) leaves the model's weights untouched. Instead, at question time it
searches the documents (e.g. with embeddings + a vector store) for the most relevant passages and stuffs them
into the prompt as context, then asks the model to answer using only that context. It's fast to set up and easy
to update (swap in new PDFs, no retraining), and it grounds answers in the literal retrieved text — but the
model's *behaviour* (does it cite correctly, does it refuse off-topic questions, does it hallucinate when
retrieval fails) is only as good as the prompt instructions, not something it was actually trained to do.

In short: **RAG changes what the model reads. Fine-tuning changes how the model behaves.** A production system
often combines both — RAG for up-to-date facts, fine-tuning for consistent behaviour and format.


## Submission checklist
- [ ] `train.jsonl` (already provided / built from the PDFs)
- [ ] This notebook (the fine-tuning code you ran)
- [ ] Screenshots of BEFORE vs AFTER on 3+ questions (answerable, off-topic, not-in-docs)
- [ ] The Gradio `gradio.live` demo link (or a short screen recording)
- [ ] Your fine-tuning vs RAG paragraph (Section 8, in your own words)
